In [0]:
dbutils.widgets.text("input_path", "")
dbutils.widgets.text("output_path", "")

input_path = dbutils.widgets.get("input_path").strip()
output_path = dbutils.widgets.get("output_path").strip()

In [0]:
if not input_path or not output_path:
    raise ValueError(
        "Both input_path and output_path must be set. "
        "Example input: /Volumes/dev_automotive/landing/landing_raw/catalogs.csv | "
        "Example output: /Volumes/dev_automotive/landing/landing_raw/catalogs.json"
    )

print(f"Input (CSV):  {input_path}")
print(f"Output (JSON): {output_path}")

In [0]:
# COMMAND ----------

import csv
import json
from datetime import datetime

def csv_to_json(csv_file, json_file):
    """Read CSV into list of dicts, add audit columns, write single JSON file (indent=2)."""
    try:
        with open(csv_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            data = list(reader)

        if not data:
            print("No data in the file")
            return 0

        # Audit columns (Databricks best practice)
        loaded_at = datetime.utcnow().isoformat() + "Z"
        for row in data:
            row["loaded_at"] = loaded_at
            row["source_file"] = csv_file

        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"Successfully converted {csv_file} to {json_file}")
        return len(data)

    except FileNotFoundError:
        print(f"Error: File {csv_file} not found")
        raise
    except Exception as e:
        print(f"Error: {e}")
        raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Run conversion and verify count

# COMMAND ----------

input_count = csv_to_json(input_path, output_path)
print(f"Records written to JSON: {input_count}")
